# 🛒 Sesión 2: Extracción de Grafos (Caso E-commerce)
**Objetivo:** convertir un log de compras (grafo bipartito Tickets → Productos) en una red de co-compra Producto ↔ Producto, y responder preguntas de negocio con `networkx`.

**Equipo:** _(escribir nombres aquí)_

## Fase 1: Limpieza del Log (Data Cleaning)

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

df = pd.read_csv('dataset/compras_ecommerce.csv')
print('Filas originales:', len(df))
print('Productos distintos (sin limpiar):', df['Producto'].nunique())
display(df.head())
display(df.isna().sum())

In [ ]:
# 1. Eliminar filas con producto nulo
df = df.dropna(subset=['Producto'])
print('Tras eliminar nulos:', len(df))

# 2. Estandarizar a MAYÚSCULAS (y quitar espacios sobrantes)
df['Producto'] = df['Producto'].str.strip().str.upper()
print('Productos distintos (estandarizados):', df['Producto'].nunique())

# 3. Eliminar tickets fallidos
df = df[df['ID_Transaccion'] != 'ERR_TIMEOUT']
print('Tras eliminar ERR_TIMEOUT:', len(df))

# 4. Self-loops: mismo producto repetido dentro del mismo ticket
df = df.drop_duplicates(subset=['ID_Transaccion', 'Producto'])
print('Tras eliminar duplicados por ticket:', len(df))
print('Tickets únicos:', df['ID_Transaccion'].nunique())

## Fase 2: Extracción (Colapsar el Grafo Bipartito)
Hacemos un `merge` del dataframe consigo mismo por `ID_Transaccion`: cada ticket genera todos los pares de productos que contiene. Filtramos con `Producto_A < Producto_B` (orden alfabético), lo cual elimina **a la vez** los self-loops (A = B) y los pares espejo (Mouse-Laptop vs Laptop-Mouse).

In [ ]:
cruce = pd.merge(df, df, on='ID_Transaccion', suffixes=('_A', '_B'))
print('Filas tras el merge:', len(cruce))

cruce = cruce[cruce['Producto_A'] < cruce['Producto_B']]
print('Filas tras filtrar self-loops y espejos:', len(cruce))

aristas = (cruce.groupby(['Producto_A', 'Producto_B'])
                .size()
                .reset_index(name='Peso')
                .sort_values('Peso', ascending=False)
                .reset_index(drop=True))
print('Pares únicos (aristas):', len(aristas))
display(aristas)

## Fase 3: Respuestas de Negocio

In [ ]:
G = nx.from_pandas_edgelist(aristas, source='Producto_A', target='Producto_B', edge_attr='Peso')
print('Nodos:', G.number_of_nodes(), '| Aristas:', G.number_of_edges())
print('Densidad:', round(nx.density(G), 3))

### 1. La Joya de la Corona (Degree Centrality)

In [ ]:
dc = pd.Series(nx.degree_centrality(G)).sort_values(ascending=False)
display(dc.round(3).to_frame('Degree Centrality'))
print('Producto con mayor Degree Centrality:', dc.index[0], '->', dc.iloc[0])
print('¿Todos empatados?', dc.nunique() == 1)

**Hallazgo:** la Degree Centrality vale **1.0 para los 10 productos**. El grafo es completo: cada producto se ha comprado alguna vez junto a todos los demás, así que la centralidad de grado no discrimina y no hay un hub por conteo de vecinos. Para desempatar usamos la **fuerza ponderada** (suma de los pesos de las aristas de cada nodo), que sí considera cuántas veces ocurre cada co-compra.

In [ ]:
fuerza = pd.Series(dict(G.degree(weight='Peso'))).sort_values(ascending=False)
display(fuerza.to_frame('Fuerza (suma de pesos)'))
print('PRODUCTO ANCLA (por fuerza ponderada):', fuerza.index[0])

**Veredicto 1:** el producto ancla para la portada es la **WEBCAM 1080P** (1,721), seguida del **MONITOR 27" 144HZ** (1,648). Matiz: la fuerza ponderada también refleja que la webcam es el producto más vendido.

### 2. El Algoritmo en Acción: recomendación para el teclado

In [ ]:
objetivo = 'TECLADO MECÁNICO RGB'
vecinos = sorted(((v, d['Peso']) for v, d in G[objetivo].items()), key=lambda x: x[1], reverse=True)
display(pd.DataFrame(vecinos, columns=['Producto vecino', 'Peso']))
print(f'RECOMENDAR a quien compra {objetivo}: {vecinos[0][0]} (peso {vecinos[0][1]})')

**Veredicto 2:** recomendar **MOUSEPAD XL** (peso 359). La silla gamer (351) y el monitor (350) quedan casi empatados como opciones secundarias; el resto de los vecinos tiene pesos de 26 a 37.

### 3. Renderizado (grosor de línea ∝ Peso)

In [ ]:
import textwrap

# Grafo dirigido solo para dibujar flechas (la dirección sigue el orden alfabético A -> B)
G_twitch = nx.from_pandas_edgelist(aristas, source='Producto_A', target='Producto_B', edge_attr='Peso', create_using=nx.DiGraph())
etiquetas = {n: textwrap.fill(n, 12) for n in G_twitch.nodes()}  # nombres largos en varias líneas

plt.figure(figsize=(14,11))
pos = nx.spring_layout(G_twitch, seed=42, k=2.5, weight=None)
pesos = [d['Peso']*0.02 for (u, v, d) in G_twitch.edges(data=True)]  # grosor proporcional al Peso
nx.draw(G_twitch, pos, labels=etiquetas, with_labels=True, node_color='#9146FF', node_size=8000, font_size=8, font_color='white', font_weight='bold', width=pesos, edge_color='gray', arrowsize=20)
plt.title('Red de Co-compra de Productos', fontsize=16)
plt.show()

**Nota sobre las flechas:** la co-compra no tiene dirección real; las flechas van de A a B en orden alfabético y solo se usan para el estilo del grafo dirigido. El grosor es `Peso × 0.02` (con `× 0.5` las líneas serían de ~180 px por pesos de ~350).\n\n**Lectura visual:** el grafo muestra dos clústeres de aristas gruesas (pesos ≈ 350) frente a un fondo de aristas finas (≈ 30):
- **Setup de escritorio:** Teclado, Mousepad, Monitor, Silla.
- **Setup de trabajo/streaming:** Webcam, Audífonos, Laptop, Mouse.

El Micrófono y el Cable HDMI quedan en la periferia. Esto confirma visualmente la recomendación del Mousepad para el teclado.

## Conclusiones
1. Limpieza: 6,858 → 6,258 filas; 2,494 tickets; 10 productos.
2. Degree Centrality = 1.0 en todos (grafo completo); desempate por fuerza ponderada → **Webcam 1080p**.
3. Para el Teclado Mecánico RGB → **Mousepad XL**.
4. Existen dos clústeres de co-compra que pueden usarse para bundles.